In [15]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import h5py
import re

root = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/refining/data_slash_results/"
filename = root + 'archs4_gene_human_v2.1_normalized_LW20230126.h5'
with h5py.File(filename, 'r') as f:
    low_cat_matrix = f['data/lowest_tier_category_matrix2'][:]
    #high_cat_matrix = f['data/highest_tier_category_matrix2'][:] # Done later
    gene_names = f['meta/genes/all_gene_names'][:]
    category_names = f['meta/biosamples/category'][:]

category_names = [x.decode('UTF-8') for x in category_names]
gene_names = [x.decode('UTF-8') for x in gene_names]

category_list  = ["Heart", "Atrium", "Cardiac Muscle Fiber", "Pericardium", "Valve", "Ventricle","HUVEC", 
    "Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
    "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF",
    "HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
    "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
    "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
    "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus","Granulocyte", "Neutrophil", "Blymphocyte", 
    "Plasma Cell", "Plasmacytoid Dendritic Cell", 
    "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
    "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
    "Thymocyte", "Spleen",
    "Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
    "A431", "HACAT", "HNSCC","Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
    "Vascular Smooth Muscle", "LHCNM2","Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
    "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
    "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron","Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
    "Trachea","Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
    "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
    "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]

category_dict  = {
    "Cardiovascular System" : ["Heart", "Atrium", "Cardiac Muscle Fiber", 
                               "Pericardium", "Valve", "Ventricle",
                               "HUVEC"], 
    
    "Connective Tissue" : ["Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
                           "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF"],

    "Digestive System": ["HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
                         "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
                         "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
                         "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus"],
    
    "Immune System" : ["Granulocyte", "Neutrophil", "Blymphocyte", "Plasma Cell", "Plasmacytoid Dendritic Cell", 
                       "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
                       "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
                       "Thymocyte", "Spleen"],
    
    "Integumentary System" : ["Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
                              "A431", "HACAT", "HNSCC"], 
    
    "Muscular System" : ["Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
                         "Vascular Smooth Muscle", "LHCNM2"], 
    
    "Nervous System" : ["Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
                        "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
                        "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron"], 
    
    "Respiratory System" : ["Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
                            "Trachea"], 
    
    "Urogenital System" : ["Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
                           "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
                           "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]
}

In [16]:
from IPython.display import display

df = pd.DataFrame(low_cat_matrix, index=gene_names)
df.columns = category_list
#display(df)

---
# Calculations

In [17]:

# Compute row-wise averages while skipping NaN values
row_averages = np.nanmean(low_cat_matrix, axis=1)

# Create new matrix where each value is divided by row average
low_cat_oe_matrix = np.divide(low_cat_matrix, row_averages[:, np.newaxis], 
                       out=np.zeros_like(low_cat_matrix), where=row_averages[:, np.newaxis]!=0)

print("Low Cat matrix shape:")
print(low_cat_matrix.shape)
print(low_cat_matrix[0][:5])
print()
print("Average:", row_averages[0])
print()
print("Low Cat OE matrix shape:")
print(low_cat_oe_matrix.shape)
print(low_cat_oe_matrix[0][:5])

Low Cat matrix shape:
(57447, 127)
[0.05616796 0.04883453        nan 0.02061483 0.03719162]

Average: 0.09820224169868699

Low Cat OE matrix shape:
(57447, 127)
[0.57196205 0.49728533        nan 0.20992223 0.37872471]


In [18]:
low_cat_log2_oe_matrix = np.log2(low_cat_oe_matrix)

In [19]:
from IPython.display import display

df = pd.DataFrame(low_cat_matrix, index=gene_names)#low_cat_log2_oe_matrix
df.columns = category_list
#display(df)

---
# Results

In [20]:
# loop through each column and get the top 10 highest values and their indexes
df = pd.DataFrame(low_cat_matrix, index=gene_names)
df.columns = category_list

for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print('Indexes for Column "{}":'.format(col))
    print(top_10.index)
    print()

Top 10 values in Column "Heart":
TPM1               5.742830
TNNT2              5.338282
ACTC1              5.300183
AHNAK              5.196548
GAPDH              5.145582
RPL41              5.069228
TNNC1              4.838087
ENSG00000283907    4.796245
MYH7               4.612605
MYL2               4.601054
Name: Heart, dtype: float64
Indexes for Column "Heart":
Index(['TPM1', 'TNNT2', 'ACTC1', 'AHNAK', 'GAPDH', 'RPL41', 'TNNC1',
       'ENSG00000283907', 'MYH7', 'MYL2'],
      dtype='object')

Top 10 values in Column "Atrium":
NPPA      8.230254
MYL7      7.551266
ACTC1     6.878828
MYH6      6.852111
TNNT2     6.524649
TPM1      6.114578
MYL4      6.088424
DES       5.944048
ANKRD1    5.900489
CRYAB     5.670695
Name: Atrium, dtype: float64
Indexes for Column "Atrium":
Index(['NPPA', 'MYL7', 'ACTC1', 'MYH6', 'TNNT2', 'TPM1', 'MYL4', 'DES',
       'ANKRD1', 'CRYAB'],
      dtype='object')

Top 10 values in Column "Cardiac Muscle Fiber":
A1BG        NaN
A1BG-AS1    NaN
A1CF        

Top 10 values in Column "Gastric Epithelial Cell":
FTH1      7.189936
GAPDH     6.326608
TMSB10    5.770218
ACTB      5.657337
RPL41     5.612830
RPS18     5.551259
RPL18A    5.367057
RPS2      5.360098
FTL       5.237635
RPL13A    5.158275
Name: Gastric Epithelial Cell, dtype: float64
Indexes for Column "Gastric Epithelial Cell":
Index(['FTH1', 'GAPDH', 'TMSB10', 'ACTB', 'RPL41', 'RPS18', 'RPL18A', 'RPS2',
       'FTL', 'RPL13A'],
      dtype='object')

Top 10 values in Column "Esophagus":
RPL41     5.598479
RPL13     5.499775
GAPDH     5.478528
RPS2      5.365995
RPS18     5.062279
RPL7A     4.902571
FTH1      4.900557
ACTB      4.829408
RPL13A    4.798024
RPS8      4.797469
Name: Esophagus, dtype: float64
Indexes for Column "Esophagus":
Index(['RPL41', 'RPL13', 'GAPDH', 'RPS2', 'RPS18', 'RPL7A', 'FTH1', 'ACTB',
       'RPL13A', 'RPS8'],
      dtype='object')

Top 10 values in Column "Granulocyte":
B2M       6.644132
S100A9    6.619018
S100A8    6.138884
FTL       5.668644
FTH1      

Top 10 values in Column "Respiratory Smooth Muscle":
A1BG        NaN
A1BG-AS1    NaN
A1CF        NaN
A2M         NaN
A2M-AS1     NaN
A2ML1       NaN
A2ML1-AS1   NaN
A2ML1-AS2   NaN
A2MP1       NaN
A3GALT2     NaN
Name: Respiratory Smooth Muscle, dtype: float64
Indexes for Column "Respiratory Smooth Muscle":
Index(['A1BG', 'A1BG-AS1', 'A1CF', 'A2M', 'A2M-AS1', 'A2ML1', 'A2ML1-AS1',
       'A2ML1-AS2', 'A2MP1', 'A3GALT2'],
      dtype='object')

Top 10 values in Column "Vascular Smooth Muscle":
GAPDH     6.441466
FTL       6.215082
COL1A1    6.074400
RPL41     5.999033
TMSB4X    5.927418
FTH1      5.860288
VIM       5.816866
AHNAK     5.767804
TMSB10    5.726166
COL1A2    5.722030
Name: Vascular Smooth Muscle, dtype: float64
Indexes for Column "Vascular Smooth Muscle":
Index(['GAPDH', 'FTL', 'COL1A1', 'RPL41', 'TMSB4X', 'FTH1', 'VIM', 'AHNAK',
       'TMSB10', 'COL1A2'],
      dtype='object')

Top 10 values in Column "LHCNM2":
GAPDH     7.142394
FTH1      6.846385
TMSB10    6.546708
FTL 

Top 10 values in Column "T47D":
RPL41     7.228656
AHNAK     6.678778
RPS18     6.337045
TMSB4X    6.206490
RPS24     5.822256
RPL7A     5.818329
ACTB      5.755012
KRT8      5.648793
RPL13     5.583629
RPL35A    5.518266
Name: T47D, dtype: float64
Indexes for Column "T47D":
Index(['RPL41', 'AHNAK', 'RPS18', 'TMSB4X', 'RPS24', 'RPL7A', 'ACTB', 'KRT8',
       'RPL13', 'RPL35A'],
      dtype='object')

Top 10 values in Column "HELA":
GAPDH     6.520240
FTH1      6.055026
RPL41     5.825150
RPS18     5.596109
RPS2      5.395286
RPL7A     5.289063
RPS17     5.197331
EEF1A1    5.151923
ENO1      5.077786
RPL19     5.021565
Name: HELA, dtype: float64
Indexes for Column "HELA":
Index(['GAPDH', 'FTH1', 'RPL41', 'RPS18', 'RPS2', 'RPL7A', 'RPS17', 'EEF1A1',
       'ENO1', 'RPL19'],
      dtype='object')

Top 10 values in Column "Kidney":
RPL41    5.820648
GAPDH    5.743612
RPL8     5.485636
RPL13    5.434437
RPS2     5.392780
RPS18    5.310937
ACTB     5.207562
FTL      5.182444
ENO1     4.97109

In [21]:
# loop through each column and get the top 10 highest values and their indexes

df = pd.DataFrame(low_cat_oe_matrix, index=gene_names)#low_cat_log2_oe_matrix
df.columns = category_list

for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print('Indexes for Column "{}":'.format(col))
    print(top_10.index)
    print()

Top 10 values in Column "Heart":
RN7SKP288          60.015807
CACNA1C-IT1        59.236888
ENSG00000286639    49.689579
ENSG00000267784    49.565267
BTF3L4P4           45.925513
ENSG00000232056    45.245420
ENSG00000238156    43.591272
ENSG00000250500    42.585218
ENSG00000271401    37.209561
ENSG00000237324    36.547886
Name: Heart, dtype: float64
Indexes for Column "Heart":
Index(['RN7SKP288', 'CACNA1C-IT1', 'ENSG00000286639', 'ENSG00000267784',
       'BTF3L4P4', 'ENSG00000232056', 'ENSG00000238156', 'ENSG00000250500',
       'ENSG00000271401', 'ENSG00000237324'],
      dtype='object')

Top 10 values in Column "Atrium":
BMP10              92.512588
MIR208A            85.817873
ENSG00000287891    67.385660
MYBPHL             66.685877
ENSG00000261606    66.180544
ENSG00000225519    64.210066
LMAN1L             60.449920
MYH6               57.154627
LINC01880          54.779973
LINC02248          52.474549
Name: Atrium, dtype: float64
Indexes for Column "Atrium":
Index(['BMP10', 'MIR2

Top 10 values in Column "Blymphocyte":
ENSG00000236864    55.688606
LINC01991          49.174252
PSMA8              46.358556
HTR3A              45.652595
SERPINA9           44.003326
SPATA48            43.408532
LRRTM4-AS1         41.622893
ENSG00000260030    40.817126
PRAMENP            39.890129
RGS13              39.054651
Name: Blymphocyte, dtype: float64
Indexes for Column "Blymphocyte":
Index(['ENSG00000236864', 'LINC01991', 'PSMA8', 'HTR3A', 'SERPINA9', 'SPATA48',
       'LRRTM4-AS1', 'ENSG00000260030', 'PRAMENP', 'RGS13'],
      dtype='object')

Top 10 values in Column "Plasma Cell":
IGHV3OR16-11    82.643455
IGHV3OR16-15    73.434339
IGHV3OR16-13    71.446504
IGHV3-60        71.398107
IGHV3-62        67.336964
IGHV3OR16-17    65.158567
IGLV3-17        65.129226
RNU6-1135P      65.022916
IGHV3-22        63.939817
IGHV3OR16-8     63.385570
Name: Plasma Cell, dtype: float64
Indexes for Column "Plasma Cell":
Index(['IGHV3OR16-11', 'IGHV3OR16-15', 'IGHV3OR16-13', 'IGHV3-60', 'IGHV

Top 10 values in Column "SKNSH":
ENSG00000213328    63.397441
PCDHA8             55.683877
TP53TG3HP          53.908791
PHOX2A             51.427785
MFFP3              49.528555
ENSG00000278009    47.338010
ENSG00000279239    43.861289
DBH                43.680099
RN7SL390P          42.273823
LINC02382          42.126959
Name: SKNSH, dtype: float64
Indexes for Column "SKNSH":
Index(['ENSG00000213328', 'PCDHA8', 'TP53TG3HP', 'PHOX2A', 'MFFP3',
       'ENSG00000278009', 'ENSG00000279239', 'DBH', 'RN7SL390P', 'LINC02382'],
      dtype='object')

Top 10 values in Column "Astrocyte":
ENSG00000231827    24.902209
PI15               16.837557
GSX1               16.485285
PAX7               16.368538
LINC00404          14.811923
SOX3               14.298406
ENSG00000276074    12.990924
FAM181A            11.420357
LINC01159          10.884474
POU3F4             10.816788
Name: Astrocyte, dtype: float64
Indexes for Column "Astrocyte":
Index(['ENSG00000231827', 'PI15', 'GSX1', 'PAX7', 'LINC00404

Top 10 values in Column "C42":
KLK3               62.671097
KLK2               54.638844
MIR1279            49.514490
PAGE1              38.334180
ENSG00000252798    36.371418
ENSG00000288678    32.546169
KLKP1              31.259949
KLK15              30.595513
KLK4               29.232707
ENSG00000252691    28.962468
Name: C42, dtype: float64
Indexes for Column "C42":
Index(['KLK3', 'KLK2', 'MIR1279', 'PAGE1', 'ENSG00000252798',
       'ENSG00000288678', 'KLKP1', 'KLK15', 'KLK4', 'ENSG00000252691'],
      dtype='object')

Top 10 values in Column "DU145":
RPL31P11           54.145440
SFTA3              26.716879
NKX2-1-AS1         17.523533
ENSG00000218048    16.431916
RPL36AP15          16.082133
CDRT15P12          15.836594
CT83               15.221197
ENSG00000288101    14.437143
RPSAP71            12.508291
LINC00707          12.501351
Name: DU145, dtype: float64
Indexes for Column "DU145":
Index(['RPL31P11', 'SFTA3', 'NKX2-1-AS1', 'ENSG00000218048', 'RPL36AP15',
       'CDRT15P12

In [22]:
# loop through each column and get the top 10 highest values and their indexes

df = pd.DataFrame(low_cat_log2_oe_matrix, index=gene_names)#low_cat_log2_oe_matrix
df.columns = category_list


for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print('Indexes for Column "{}":'.format(col))
    print(top_10.index)
    print()

Top 10 values in Column "Heart":
RN7SKP288          5.907271
CACNA1C-IT1        5.888424
ENSG00000286639    5.634871
ENSG00000267784    5.631258
BTF3L4P4           5.521224
ENSG00000232056    5.499700
ENSG00000238156    5.445967
ENSG00000250500    5.412281
ENSG00000271401    5.217601
ENSG00000237324    5.191716
Name: Heart, dtype: float64
Indexes for Column "Heart":
Index(['RN7SKP288', 'CACNA1C-IT1', 'ENSG00000286639', 'ENSG00000267784',
       'BTF3L4P4', 'ENSG00000232056', 'ENSG00000238156', 'ENSG00000250500',
       'ENSG00000271401', 'ENSG00000237324'],
      dtype='object')

Top 10 values in Column "Atrium":
BMP10              6.531578
MIR208A            6.423206
ENSG00000287891    6.074370
MYBPHL             6.059309
ENSG00000261606    6.048335
ENSG00000225519    6.004728
LMAN1L             5.917669
MYH6               5.836798
LINC01880          5.775577
LINC02248          5.713546
Name: Atrium, dtype: float64
Indexes for Column "Atrium":
Index(['BMP10', 'MIR208A', 'ENSG000002878

Top 10 values in Column "Neutrophil":
CXCR1              5.530712
ENSG00000254789    5.461188
LINC02555          5.452821
CEACAM3            5.429004
FCGR3B             5.379058
RPL9P33            5.345715
CXCR2              5.328354
TREML5P            5.317428
ENSG00000259962    5.286903
ADGRE3             5.267164
Name: Neutrophil, dtype: float64
Indexes for Column "Neutrophil":
Index(['CXCR1', 'ENSG00000254789', 'LINC02555', 'CEACAM3', 'FCGR3B', 'RPL9P33',
       'CXCR2', 'TREML5P', 'ENSG00000259962', 'ADGRE3'],
      dtype='object')

Top 10 values in Column "Blymphocyte":
ENSG00000236864    5.799310
LINC01991          5.619831
PSMA8              5.534764
HTR3A              5.512625
SERPINA9           5.459541
SPATA48            5.439907
LRRTM4-AS1         5.379305
ENSG00000260030    5.351103
PRAMENP            5.317960
RGS13              5.287422
Name: Blymphocyte, dtype: float64
Indexes for Column "Blymphocyte":
Index(['ENSG00000236864', 'LINC01991', 'PSMA8', 'HTR3A', 'SERPINA9', 

Top 10 values in Column "Respiratory Smooth Muscle":
A1BG        NaN
A1BG-AS1    NaN
A1CF        NaN
A2M         NaN
A2M-AS1     NaN
A2ML1       NaN
A2ML1-AS1   NaN
A2ML1-AS2   NaN
A2MP1       NaN
A3GALT2     NaN
Name: Respiratory Smooth Muscle, dtype: float64
Indexes for Column "Respiratory Smooth Muscle":
Index(['A1BG', 'A1BG-AS1', 'A1CF', 'A2M', 'A2M-AS1', 'A2ML1', 'A2ML1-AS1',
       'A2ML1-AS2', 'A2MP1', 'A3GALT2'],
      dtype='object')

Top 10 values in Column "Vascular Smooth Muscle":
ENSG00000253308    5.361874
LCN9               5.305324
STATH              4.937458
EVX1-AS            4.535436
NBPF7P             4.419099
ENSG00000274702    4.210476
EGFL6              4.195786
EVX1               4.187787
OSTN               4.043792
SLC39A2            4.017922
Name: Vascular Smooth Muscle, dtype: float64
Indexes for Column "Vascular Smooth Muscle":
Index(['ENSG00000253308', 'LCN9', 'STATH', 'EVX1-AS', 'NBPF7P',
       'ENSG00000274702', 'EGFL6', 'EVX1', 'OSTN', 'SLC39A2'],
     

Top 10 values in Column "Kidney":
LINC02121    5.044086
LINC02294    4.568717
ATP6V1G3     4.522475
TMEM174      4.453135
NPHS2        4.433253
TMEM207      4.403540
LINC02061    4.208533
FXYD4        4.208331
SLC7A13      4.188309
SLC22A8      4.184220
Name: Kidney, dtype: float64
Indexes for Column "Kidney":
Index(['LINC02121', 'LINC02294', 'ATP6V1G3', 'TMEM174', 'NPHS2', 'TMEM207',
       'LINC02061', 'FXYD4', 'SLC7A13', 'SLC22A8'],
      dtype='object')

Top 10 values in Column "Podocyte":
OR12D2             5.592075
DCAF12L1           4.687778
LRRN4              4.548477
RNU4-66P           4.409631
OR2G1P             4.321238
ENSG00000267509    3.932931
GDF5               3.922411
ACTBL2             3.917705
PAX8               3.906463
LINC01173          3.728246
Name: Podocyte, dtype: float64
Indexes for Column "Podocyte":
Index(['OR12D2', 'DCAF12L1', 'LRRN4', 'RNU4-66P', 'OR2G1P', 'ENSG00000267509',
       'GDF5', 'ACTBL2', 'PAX8', 'LINC01173'],
      dtype='object')

Top 10 val

---

In [23]:
with h5py.File(filename, 'r') as f:
    high_cat_matrix = f['data/highest_tier_category_matrix2'][:]


In [24]:

# Compute row-wise averages while skipping NaN values
row_averages = np.nanmean(high_cat_matrix, axis=1)

# Create new matrix where each value is divided by row average
high_cat_oe_matrix = np.divide(high_cat_matrix, row_averages[:, np.newaxis], 
                       out=np.zeros_like(high_cat_matrix), where=row_averages[:, np.newaxis]!=0)

print("High Cat matrix shape:")
print(high_cat_matrix.shape)
print(high_cat_matrix[0][:5])
print()
print("Average:", row_averages[0])
print()
print("High Cat OE matrix shape:")
print(high_cat_oe_matrix.shape)
print(high_cat_oe_matrix[0][:5])

High Cat matrix shape:
(57447, 9)
[0.03641399 0.12006714 0.4371865  0.13372832 0.09514402]

Average: 0.12443487557940484

High Cat OE matrix shape:
(57447, 9)
[0.29263489 0.96489946 3.51337598 1.07468517 0.76460899]


In [25]:
from IPython.display import display

df = pd.DataFrame(high_cat_oe_matrix, index=gene_names)
df.columns = ["Cardiovascular System",  "Connective Tissue", "Digestive System", 
"Immune System", "Integumentary System", "Muscular System", "Nervous System", 
"Respiratory System", "Urogenital System"]
display(df)

,Cardiovascular System,Connective Tissue,Digestive System,Immune System,Integumentary System,Muscular System,Nervous System,Respiratory System,Urogenital System
A1BG,0.292635,0.964899,3.513376,1.074685,0.764609,0.610123,0.649582,0.585810,0.544280
A1BG-AS1,0.531534,0.862849,0.587254,1.686029,1.248356,1.181383,0.851127,0.941672,1.109794
A1CF,0.042559,0.128310,8.103669,0.062638,0.070555,0.061019,0.058519,0.256538,0.216194
A2M,1.453439,0.630875,2.134600,1.279560,0.554727,0.961388,0.872091,0.781774,0.331546
A2M-AS1,1.706245,0.896285,1.418268,1.335137,0.385462,1.177389,0.827398,0.789855,0.463960
...,...,...,...,...,...,...,...,...,...
ZYG11B,1.186541,0.804524,0.726652,0.707691,0.793708,2.257337,1.249017,0.614579,0.659952
ZYX,1.178191,0.885927,0.836426,1.354094,1.206446,0.954025,0.731370,1.067979,0.785542
ZYXP1,0.643183,1.664882,0.982061,0.908192,1.027365,0.721292,0.697635,0.893924,1.461466
ZZEF1,1.078781,0.933153,0.961537,1.334507,0.916977,1.023628,0.953668,0.914227,0.883521


In [26]:
# loop through each column and get the top 10 highest values and their indexes
for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print('Indexes for Column "{}":'.format(col))
    print(top_10.index)
    print()

Top 10 values in Column "Cardiovascular System":
MYBPC3             8.470539
ENSG00000236958    8.351298
MYL7               8.236753
ENSG00000286800    8.219700
ENSG00000287891    8.186218
NPPA               8.147898
LINC02502          8.103888
ENSG00000286820    8.073964
TNNI3K             7.764016
ENSG00000286598    7.692011
Name: Cardiovascular System, dtype: float64
Indexes for Column "Cardiovascular System":
Index(['MYBPC3', 'ENSG00000236958', 'MYL7', 'ENSG00000286800',
       'ENSG00000287891', 'NPPA', 'LINC02502', 'ENSG00000286820', 'TNNI3K',
       'ENSG00000286598'],
      dtype='object')

Top 10 values in Column "Connective Tissue":
LINC02573          8.243832
ENSG00000287973    8.194014
ENSG00000237311    8.117703
RHAG               8.083788
ABCB10P4           8.006390
HBBP1              7.996683
HBZ                7.980091
GATA1              7.937128
ENSG00000264334    7.889233
LINC01029          7.878699
Name: Connective Tissue, dtype: float64
Indexes for Column "Connectiv

In [ ]:
Top 10 values in Column "Urogenital System":
UMOD               3.049670
DSCAM-AS1          2.985573
ENSG00000223808    2.949586
NPHS2              2.917705
OOSP2              2.904903
DPPA5              2.856769
TMEM174            2.841081
H1-8               2.821763
RNU6-813P          2.821429
NLRP13             2.792411
Name: Urogenital System, dtype: float64

Gene descriptions for Column "Urogenital System":
uromodulin
DSCAM antisense RNA 1
novel transcript
NPHS2 stomatin family member, podocin
oocyte secreted protein 2
developmental pluripotency associated 5
transmembrane protein 174
H1.8 linker histone
RNA, U6 small nuclear 813, pseudogene
NLR family pyrin domain containing 13